# LUSI vs. Kernel Ridge Regression: Baseline Experiment

This notebook investigates Learning Using Statistical Invariants
(LUSI) using a kernel-based formulation from Liu et al. (2025).

Objectives:
1. Implement the LUSI kernel solution.
2. Compare LUSI against Kernel Ridge Regression (KRR).
3. Verify numerical equivalence when tau = 0.
4. Investigate sensitivity to the invariant weight tau.
5. Visualize decision boundaries.

Dataset: Synthetic make_moons classification data.

Reference:
Liu, Y., Fokoué, E., & Krutz, D. E. (2025).
Learning using statistical invariants: when they work and when they don't.
https://doi.org/10.1007/s10994-025-06789-y

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

from sklearn.datasets import make_moons
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RBF_GAMMA = 0.5
ETA = 0.1
LUSI_GAMMA = 1.0

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def fit_lusi(
    X,
    y,
    lusi_gamma=1.0,
    tau=1.0,
    eta=0.1,
    rbf_gamma=0.5
):
    """
    LUSI kernel solution from Liu et al. (2025), Eq. 1.

    Uses constant and linear statistical predicates.
    """
    n = len(X)

    K = rbf_kernel(X, gamma=rbf_gamma)

    Phi = np.column_stack([
        np.ones(n),
        X
    ])

    P = Phi @ Phi.T

    W = lusi_gamma * np.eye(n) + tau * P

    alpha = np.linalg.solve(
        W @ K + eta * np.eye(n),
        W @ y
    )

    return alpha


def classify(scores):
    """Convert continuous prediction scores into {-1, 1}."""
    return np.where(scores >= 0, 1, -1)

In [ ]:
X, y = make_moons(
    n_samples=100,
    noise=0.15,
    random_state=RANDOM_STATE
)

# Convert labels from {0, 1} to {-1, 1}
y = 2 * y - 1

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print(
    "Training labels:",
    np.unique(y_train, return_counts=True)
)

print(
    "Testing labels:",
    np.unique(y_test, return_counts=True)
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="coolwarm",
    edgecolors="black",
    s=50,
    alpha=0.85
)

ax.set_title("Make Moons: Training Dataset")
ax.set_xlabel("Standardized Feature 1")
ax.set_ylabel("Standardized Feature 2")

fig.tight_layout()
plt.show()

In [ ]:
alpha = fit_lusi(
    X_train,
    y_train,
    lusi_gamma=LUSI_GAMMA,
    tau=1.0,
    eta=ETA,
    rbf_gamma=RBF_GAMMA
)

krr = KernelRidge(
    alpha=ETA,
    kernel="rbf",
    gamma=RBF_GAMMA
)

krr.fit(X_train, y_train)

K_train = rbf_kernel(
    X_train,
    X_train,
    gamma=RBF_GAMMA
)

K_test = rbf_kernel(
    X_test,
    X_train,
    gamma=RBF_GAMMA
)

print("Kernel symmetric:", np.allclose(K_train, K_train.T))
print("Minimum eigenvalue:", np.linalg.eigvalsh(K_train).min())
print("Coefficient shape:", alpha.shape)
print("All coefficients finite:", np.isfinite(alpha).all())

In [ ]:
lusi_train_scores = K_train @ alpha
lusi_test_scores = K_test @ alpha

krr_train_scores = krr.predict(X_train)
krr_test_scores = krr.predict(X_test)

lusi_train_pred = classify(lusi_train_scores)
lusi_test_pred = classify(lusi_test_scores)

krr_train_pred = classify(krr_train_scores)
krr_test_pred = classify(krr_test_scores)

results = pd.DataFrame({
    "Model": ["LUSI", "KRR"],
    "Training Accuracy": [
        accuracy_score(y_train, lusi_train_pred),
        accuracy_score(y_train, krr_train_pred)
    ],
    "Test Accuracy": [
        accuracy_score(y_test, lusi_test_pred),
        accuracy_score(y_test, krr_test_pred)
    ]
})

display(results)

print(
    "Test prediction disagreements:",
    np.sum(lusi_test_pred != krr_test_pred)
)

print(
    "Maximum score difference:",
    np.max(np.abs(lusi_test_scores - krr_test_scores))
)

print(
    "Mean score difference:",
    np.mean(np.abs(lusi_test_scores - krr_test_scores))
)

In [ ]:
alpha_zero = fit_lusi(
    X_train,
    y_train,
    lusi_gamma=LUSI_GAMMA,
    tau=0.0,
    eta=ETA,
    rbf_gamma=RBF_GAMMA
)

lusi_zero_scores = K_test @ alpha_zero

max_difference = np.max(
    np.abs(lusi_zero_scores - krr_test_scores)
)

equivalent = np.allclose(
    lusi_zero_scores,
    krr_test_scores,
    rtol=1e-10,
    atol=1e-10
)

print("Maximum numerical difference:", max_difference)
print("Numerically equivalent:", equivalent)

assert equivalent, "LUSI (tau=0) does not match KRR."

In [ ]:
tau_values = [
    0.0,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
]

sensitivity_results = []

for tau in tau_values:

    alpha_tau = fit_lusi(
        X_train,
        y_train,
        lusi_gamma=LUSI_GAMMA,
        tau=tau,
        eta=ETA,
        rbf_gamma=RBF_GAMMA
    )

    scores_tau = K_test @ alpha_tau
    predictions_tau = classify(scores_tau)

    sensitivity_results.append({
        "tau": tau,
        "accuracy": accuracy_score(
            y_test,
            predictions_tau
        ),
        "mean_score_difference": np.mean(
            np.abs(scores_tau - krr_test_scores)
        ),
        "disagreements": np.sum(
            predictions_tau != krr_test_pred
        )
    })

df_sensitivity = pd.DataFrame(sensitivity_results)

display(df_sensitivity)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df_sensitivity["tau"],
    df_sensitivity["mean_score_difference"],
    marker="o",
    linewidth=2,
    markersize=7
)

# Symlog allows tau=0 and positive values on one axis.
ax.set_xscale("symlog", linthresh=0.01)

ax.set_xticks(tau_values)
ax.set_xticklabels([str(t) for t in tau_values])

ax.set_xlabel("Statistical Invariant Weight (tau)")
ax.set_ylabel("Mean Absolute Score Difference vs. KRR")
ax.set_title("LUSI Sensitivity to Statistical Invariant Weight")

ax.grid(True, alpha=0.3)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "lusi_tau_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR / "lusi_tau_sensitivity.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
x_min = X_train[:, 0].min() - 0.8
x_max = X_train[:, 0].max() + 0.8

y_min = X_train[:, 1].min() - 0.8
y_max = X_train[:, 1].max() + 0.8

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250)
)

grid = np.column_stack([
    xx.ravel(),
    yy.ravel()
])

# LUSI scores on grid
K_grid = rbf_kernel(
    grid,
    X_train,
    gamma=RBF_GAMMA
)

lusi_grid_scores = K_grid @ alpha

# KRR scores on grid
krr_grid_scores = krr.predict(grid)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharex=True,
    sharey=True
)

models = [
    ("LUSI (tau=1)", lusi_grid_scores),
    ("Kernel Ridge Regression", krr_grid_scores)
]

for ax, (name, scores) in zip(axes, models):

    Z = scores.reshape(xx.shape)

    ax.contourf(
        xx,
        yy,
        Z,
        levels=[-np.inf, 0, np.inf],
        cmap="coolwarm",
        alpha=0.30
    )

    ax.contour(
        xx,
        yy,
        Z,
        levels=[0],
        colors="black",
        linewidths=1.5
    )

    ax.scatter(
        X_train[:, 0],
        X_train[:, 1],
        c=y_train,
        cmap="coolwarm",
        edgecolors="black",
        s=35
    )

    ax.set_title(name)
    ax.set_xlabel("Standardized Feature 1")

axes[0].set_ylabel("Standardized Feature 2")

fig.suptitle(
    "LUSI vs. KRR: Decision Boundary Comparison",
    fontsize=14
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "lusi_krr_decision_boundaries.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURE_DIR / "lusi_krr_decision_boundaries.pdf",
    bbox_inches="tight"
)

plt.show()

## Interpretation and Research Implications

### Baseline findings

LUSI and Kernel Ridge Regression achieved identical classification
accuracy on the synthetic make_moons dataset:

- Training accuracy: 98.57%
- Test accuracy: 96.67%
- Test prediction disagreements: 0

The models produced slightly different continuous prediction scores,
indicating that the statistical-invariant contribution modifies
the fitted solution without changing classifications in this experiment.

### Numerical validation

Setting tau = 0 recovered the ordinary KRR solution to floating-point
precision, supporting the correctness of the implementation under
the tested parameter settings.

### Tau sensitivity

Increasing tau changed continuous prediction scores, but classification
accuracy remained unchanged. Score differences appeared to saturate
at larger tau values.

### Limitations

This experiment uses one synthetic dataset and a single 70/30 split.
It does not establish statistical equivalence or comparative superiority.

The predicates are constructed from the model's existing input features,
rather than independent privileged teacher information.

The tau sensitivity analysis uses a fixed holdout set and should not
be treated as hyperparameter optimization.

### Next steps for LUCY-SVM

1. Repeated stratified cross-validation.
2. Evaluation across multiple benchmark datasets.
3. Investigation of richer statistical predicates.
4. Composite-kernel integration.
5. Comparison against LUSI, KRR, and conventional SVM baselines.